# Week 7 — Feature Selection, PCA and Explained Variance
**Netflix Contents Patterns Analysis**  
**Sahana Viyani A. | 25BAD092 | AI & DS**  
**Kumaraguru College of Technology**

## Aim
Reduce the number of numerical features describing Netflix titles and visualize how much variance the principal components retain. PCA does **not** measure binge-watching or popularity.

## 1. Load dataset and prepare features

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
sns.set_theme(style="whitegrid")
# Put netflix_titles.csv beside this notebook, or update DATA_PATH.
DATA_PATH = Path("netflix_titles.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Place the original netflix_titles.csv in the same folder as this notebook.")
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
display(df.head())
print("Missing values:")
print(df.isna().sum())

In [ ]:
# Feature engineering from Netflix catalog metadata (not viewer watch histories)
data = df.copy()
for col in ['type','rating','listed_in','country','duration']:
    if col not in data.columns: data[col] = np.nan
for col in ['release_year','date_added']:
    if col not in data.columns: data[col] = np.nan
# Dates sometimes have extra spaces; invalid dates become missing.
data['date_added_clean'] = pd.to_datetime(data['date_added'].astype('string').str.strip(), errors='coerce')
data['year_added'] = data['date_added_clean'].dt.year
data['month_added'] = data['date_added_clean'].dt.month
# Extract numeric duration and separate units so minutes and seasons aren't compared as equivalent.
data['duration_value'] = pd.to_numeric(data['duration'].astype('string').str.extract(r'(\d+)', expand=False), errors='coerce')
data['duration_unit'] = np.select([data['duration'].astype('string').str.contains('min', case=False, na=False), data['duration'].astype('string').str.contains('season', case=False, na=False)], ['Minutes','Seasons'], default='Unknown')
data['genre_primary'] = data['listed_in'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
data['country_primary'] = data['country'].fillna('Unknown').astype(str).str.split(',').str[0].str.strip()
data['genre_count'] = data['listed_in'].fillna('').astype(str).apply(lambda x: len([v for v in x.split(',') if v.strip()]))
data['country_count'] = data['country'].fillna('').astype(str).apply(lambda x: len([v for v in x.split(',') if v.strip()]))
data['release_year'] = pd.to_numeric(data['release_year'], errors='coerce')
data['content_age_at_addition'] = (data['year_added'] - data['release_year']).clip(lower=0)
print('Engineered features:')
display(data[['type','genre_primary','duration_value','duration_unit','genre_count','country_count','year_added','content_age_at_addition']].head())

## 2. Feature selection
Choose numeric catalog attributes and check redundancy using correlation. Missing values are filled using median.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
features = ['release_year','year_added','month_added','duration_value','genre_count','country_count','content_age_at_addition']
X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(data[features]), columns=features)
print('Before selection:', X.shape)
plt.figure(figsize=(9,6))
sns.heatmap(X.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature correlation'); plt.tight_layout(); plt.show()
selector = VarianceThreshold(threshold=0)
selected = X.loc[:, selector.fit(X).get_support()]
print('Selected features:', list(selected.columns))

## 3. Handle skewness and scale before PCA
Log-transform positively skewed non-negative count/duration columns. Standardization ensures PCA is not dominated by large units.

In [ ]:
for col in ['duration_value','genre_count','country_count','content_age_at_addition']:
    if col in selected and selected[col].skew() > 0.75 and (selected[col]>=0).all():
        selected[col] = np.log1p(selected[col])
X_scaled = StandardScaler().fit_transform(selected)
print('Scaled matrix:', X_scaled.shape)

## 4. Apply PCA

In [ ]:
pca = PCA()
components = pca.fit_transform(X_scaled)
variance = pca.explained_variance_ratio_
cumulative = np.cumsum(variance)
variance_table = pd.DataFrame({'PC':[f'PC{i+1}' for i in range(len(variance))], 'Explained variance':variance, 'Cumulative variance':cumulative})
display(variance_table.round(4))

## 5. Visualize explained and cumulative variance

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,4))
axes[0].bar(range(1,len(variance)+1), variance, color='mediumseagreen')
axes[0].set(title='Explained variance by component',xlabel='Principal component',ylabel='Proportion of variance')
axes[1].plot(range(1,len(cumulative)+1), cumulative, marker='o', color='royalblue')
axes[1].axhline(0.90, linestyle='--', color='crimson', label='90% variance')
axes[1].set(title='Cumulative explained variance',xlabel='Number of components',ylabel='Cumulative variance',ylim=(0,1.05))
axes[1].legend(); plt.tight_layout(); plt.show()
needed = np.searchsorted(cumulative,0.90)+1
print('Components needed for at least 90% variance:',needed)

## 6. Visualize first two principal components
Colors represent content type, not viewing behavior.

In [ ]:
if components.shape[1] >= 2:
    plot_data = pd.DataFrame({'PC1':components[:,0],'PC2':components[:,1], 'Type':data['type'].fillna('Unknown').astype(str)})
    plt.figure(figsize=(9,6))
    sns.scatterplot(data=plot_data, x='PC1', y='PC2', hue='Type', alpha=.55, s=35)
    plt.title('Netflix titles projected onto first two PCs'); plt.tight_layout(); plt.show()
else:
    print('Only one principal component is available; 2D scatter plot skipped.')

## 7. Interpret feature contributions (PCA loadings)

In [ ]:
loadings = pd.DataFrame(pca.components_.T, index=selected.columns, columns=[f'PC{i+1}' for i in range(pca.n_components_)])
display(loadings.iloc[:,:min(3,loadings.shape[1])].round(3))
plt.figure(figsize=(8,5))
sns.heatmap(loadings.iloc[:,:min(3,loadings.shape[1])], annot=True, cmap='RdBu_r', center=0)
plt.title('Feature loadings for first components'); plt.tight_layout(); plt.show()

## 8. Save PCA results

In [ ]:
reduced = pd.DataFrame(components[:,:needed], columns=[f'PC{i+1}' for i in range(needed)])
reduced.to_csv('netflix_week7_pca_features.csv', index=False)
variance_table.to_csv('netflix_week7_variance.csv', index=False)
print('Saved PCA features:', reduced.shape)
print('Variance retained:', round(cumulative[needed-1]*100,2), '%')

## Observation / conclusion
After running the notebook, record the variance explained by PC1 and PC2, the number of components needed to retain 90% variance, and the features with the strongest loadings. These results summarize variation in Netflix catalog attributes.